# ZS601 会议室 3DGS 几何质量提升 —— 阶段 0/1 全流程（Colab）

**使用步骤**：
1. 修改下面【配置】单元格中的 Drive 路径（数据 zip、代码 zip 在 Drive 中的位置）
2. 运行时 → 更改运行时类型 → **GPU**（T4 或更高）
3. 按顺序运行单元格：挂载 Drive → 复制 zip 到本地 → 解压 → 装依赖 → 阶段 0 预处理 → 阶段 1 冒烟 → 正式训练 → 评估对比 → 回传结果

**说明**：数据 zip 复制到 Colab 临时盘（/content）再解压，训练时从本地盘读取，避免 Drive 慢 IO。

In [ ]:
# ============ 配置（按需修改） ============
# 数据 zip 在 Drive 中的路径（我的云端硬盘/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip）
DATA_ZIP_IN_DRIVE = "/content/drive/MyDrive/LCCDataset/ZS601meetingroom/ZS601meetingroom_data.zip"
# 代码仓库（GitHub 公开仓库，v1-init2d 分支）：更新代码只需重跑「拉取代码+解压数据」单元格
REPO_URL = "https://github.com/VISjudy/ZS601_3DGS.git"
# 结果回传到 Drive 的目录
RESULTS_IN_DRIVE = "/content/drive/MyDrive/LCCDataset/ZS601meetingroom/results"

# Colab 本地工作目录（临时盘，读写快）
WORK = "/content/gs_work"
CODE_DIR = f"{WORK}/repo/gaussian-splattingWithMask"   # git clone 后代码位置
DATA_DIR = f"{WORK}/dataset"   # 解压后由下方单元格自动校正
import os; os.makedirs(WORK, exist_ok=True)
print("配置完成")

In [ ]:
# 1) 挂载 Google Drive（会弹出授权链接，按提示完成）
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash -s "$DATA_ZIP_IN_DRIVE" "$WORK"
# 2) 复制数据 zip 到 Colab 临时盘（代码改从 GitHub 仓库 v1-init2d 分支拉取，见下一单元格）
set -e
cp "$1" "$2/data.zip"
ls -lh "$2"

In [ ]:
# 3) 拉取代码（GitHub v1-init2d 分支）+ 解压数据，并自动定位数据根目录（含 sparse/0 的那一层）
# 只拉 v1-init2d 分支（v1 已验证版本，冻结）；更新代码：重跑本单元格即可
import zipfile, os, glob, shutil
# 先切到根目录：若 shell 停留在旧 repo 内（上一轮安装单元格 %cd 进去的），
# 直接删除会把当前目录删掉，导致 git clone 报 getcwd 错误
os.chdir("/")
if os.path.exists(f"{WORK}/repo"):
    shutil.rmtree(f"{WORK}/repo")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
!git clone --depth 1 -b v1-init2d "$REPO_URL" "$WORK/repo"
assert os.path.isdir(f"{WORK}/repo/gaussian-splattingWithMask"), "clone 结果缺少 gaussian-splattingWithMask 目录，请检查仓库结构"

with zipfile.ZipFile(f"{WORK}/data.zip") as zf:
    zf.extractall(f"{WORK}/data_raw")

# 自动寻找包含 sparse/0/images.txt 的目录作为 DATA_DIR
cands = glob.glob(f"{WORK}/data_raw/**/sparse/0/images.txt", recursive=True)
assert cands, "解压结果中找不到 sparse/0/images.txt，请检查 zip 结构"
DATA_DIR = os.path.dirname(os.path.dirname(os.path.dirname(cands[0])))
print("CODE_DIR =", CODE_DIR)
print("DATA_DIR =", DATA_DIR)
!ls "$DATA_DIR"

In [ ]:
# 4) 安装依赖：轻量库 + 编译 CUDA 子模块（光栅化器编译约 5~10 分钟，属正常现象）
!pip install laspy matplotlib plyfile --quiet
!cd "$CODE_DIR" && pip install ./submodules/simple-knn --quiet
!cd "$CODE_DIR" && pip install ./submodules/diff-gaussian-rasterization

# fused-ssim 用 clone 方式安装（仓库未随代码 zip 分发）；失败也不影响训练，会自动回退普通 ssim
%cd "$CODE_DIR/submodules"
!git clone https://github.com/rahul-goel/fused-ssim.git || echo "clone 失败（可选组件，自动回退普通 ssim）"
%cd fused-ssim
!pip install . --no-build-isolation || echo "fused-ssim 安装失败（可选组件，自动回退普通 ssim）"
%cd "$CODE_DIR"


## 阶段 0：数据预处理（一次性）
产物：替换后的 `points3D.txt/ply`（含颜色+法向量）、`images.txt/images_test.txt/images-val10.txt`、10 张法向量方向图（`intermediate/normal_visual/`）

In [ ]:
!cd "$CODE_DIR" && python preprocess.py --data_path "$DATA_DIR" --seed 42 --val_num 10 --test_ratio 0.05
# 验收：把 10 张法向量方向图与 GT 拼图展示
import glob
from IPython.display import display
from PIL import Image
for p in sorted(glob.glob(f"{DATA_DIR}/intermediate/normal_visual/val*_normal.png")):
    gt = p.replace("_normal.png", "_gt.png")
    a, b = Image.open(p).resize((240, 415)), Image.open(gt).resize((240, 415))
    canvas = Image.new("RGB", (485, 415))
    canvas.paste(a, (0, 0)); canvas.paste(b, (245, 0))
    display(canvas)

## 阶段 1-C① 冒烟验证（2000 轮，跑通全链路）
验收点：初始化打印各轴 scale 统计（z 轴极小）、`intermediate/init_2d/` 导出 ply、`val_render/iter_1/` 起每 500 轮渲染、`loss_log.csv` 与曲线图、`evaluate.py` 出指标。

注：`--data_device cpu` 把训练图像放 CPU 内存而非显存；`--lazy_load --lazy_cache 100` 让图片按需从磁盘读取、最多缓存 100 张，防止 2694 张图全部常驻内存导致爆 RAM（数据已放在 Colab 本地盘，读取开销很小）。

In [ ]:
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/smoke_init2d \
  --init_2d --iterations 2000 \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 2000 --save_iterations 2000 \
  --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 100 --disable_viewer

In [ ]:
# 冒烟产物检查（渲染图/法向图/GT 仅首轮、loss 记录、val 指标记录、init_2d 导出）
!ls -R "$CODE_DIR/output/smoke_init2d/val_render" | head -40
!head -3 "$CODE_DIR/output/smoke_init2d/loss_log.csv"
!cat "$CODE_DIR/output/smoke_init2d/val_metrics.csv"
!ls "$DATA_DIR/intermediate/init_2d"

from IPython.display import display
from PIL import Image
import glob, os
display(Image.open(f"{CODE_DIR}/output/smoke_init2d/loss_curves_2000.png"))
# 最后一轮 val 图对比（每组从左到右：render | gt | normal，GT 只在首轮保存）
last_dir = sorted(glob.glob(f"{CODE_DIR}/output/smoke_init2d/val_render/iter_*"),
                  key=lambda x: int(x.split('_')[-1]))[-1]
for i in range(10):
    rp = f"{last_dir}/val{i}_render.png"
    gp = f"{last_dir}/val{i}_gt.png"
    nmp = f"{last_dir}/val{i}_normal.png"
    if not os.path.exists(rp):
        continue
    paths = [q for q in [rp, gp, nmp] if os.path.exists(q)]
    imgs = [Image.open(q).resize((220, 380)) for q in paths]
    canvas = Image.new("RGB", (225 * len(imgs) - 5, 380))
    for k, im in enumerate(imgs):
        canvas.paste(im, (225 * k, 0))
    display(canvas)


In [ ]:
# 冒烟评估（图像指标 + cloud2cloud 几何指标）
!cd "$CODE_DIR" && python evaluate.py -m output/smoke_init2d -s "$DATA_DIR"

## 阶段 1-C② 正式训练（150000 轮，三组对比）
**三组训练超参完全一致，变量只有两个开关 `--init_2d`（2D 椭球初始化）与 `--freeze_2d_z`(训练中形状约束，z 轴冻结)**：
- A 组 baseline：标准 3DGS 初始化（两个开关都不开）
- B 组 init2d+freeze：2D 椭球初始化 + 训练中保持扁平（法向对齐 + z 轴冻结）
- C 组 init2d_free：仅 2D 椭球初始化，不约束形状（椭球厚度可自由优化）

⚠️ 每组需数小时。Colab 免费版可能中途断连，建议开 Pro 或用「断点续训」（`--start_checkpoint`）。
训练中途可查看 `output/*/val_render/` 的渲染图与 `val_metrics.csv` 的指标/点数/耗时变化。


In [ ]:
# A 组：baseline（标准初始化）
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/zs601_baseline \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 100 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000

In [ ]:
# B 组：init_2d + freeze_2d_z（2D 椭球初始化 + 训练中形状约束保持扁平），其余超参与 A 组完全一致
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/zs601_init2d \
  --init_2d --freeze_2d_z \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 100 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000


In [ ]:
# C 组：仅 init_2d（2D 椭球初始化，不加形状约束，椭球厚度可自由优化），其余超参与 A 组完全一致
!cd "$CODE_DIR" && python train_mask.py \
  -s "$DATA_DIR" -m output/zs601_init2d_free \
  --init_2d \
  --iterations 150000 --sh_degree 2 \
  --position_lr_init 0.000016 --position_lr_final 0.00000016 --position_lr_max_steps 150000 \
  --scaling_lr 0.0015 \
  --densification_interval 10000 --densify_until_iter 100000 \
  --opacity_reset_interval 150000 --densify_grad_threshold 0.0002 \
  --seed 42 --alpha_masks masks --data_device cpu --lazy_load --lazy_cache 100 --disable_viewer \
  --val_file "$DATA_DIR/sparse/0/images-val10.txt" \
  --test_iterations 50000 100000 150000 --save_iterations 50000 100000 150000


In [ ]:
# 三组分别评估并打印对比表
!cd "$CODE_DIR" && python evaluate.py -m output/zs601_baseline -s "$DATA_DIR"
!cd "$CODE_DIR" && python evaluate.py -m output/zs601_init2d -s "$DATA_DIR"
!cd "$CODE_DIR" && python evaluate.py -m output/zs601_init2d_free -s "$DATA_DIR"

import json
rows = []
for name in ["zs601_baseline", "zs601_init2d", "zs601_init2d_free"]:
    try:
        with open(f"{CODE_DIR}/output/{name}/metrics.json") as f:
            m = json.load(f)
    except FileNotFoundError:
        print(f"[提示] {name} 尚未训练完成，跳过")
        continue
    im, g = m["image_metrics"], m["geometry"]
    rows.append([name, f"{im['PSNR']:.3f}", f"{im['L1']:.5f}", f"{im['SSIM']:.4f}",
                 f"{g['mean']:.4f}", f"{g['median']:.4f}", f"{g['rmse']:.4f}", f"{g['p90']:.4f}"])
hdr = ["组别", "PSNR", "L1", "SSIM", "geo_mean", "geo_median", "geo_rmse", "geo_p90"]
print("\n" + " | ".join(f"{h:>16}" for h in hdr))
for r in rows:
    print(" | ".join(f"{c:>16}" for c in r))


In [ ]:
# 结果回传 Drive（排除体积大的 point_cloud ply，保留指标/曲线/渲染图；需要 ply 可自行去掉 --exclude）
!mkdir -p "$RESULTS_IN_DRIVE"
!rsync -a --exclude "point_cloud" "$CODE_DIR/output" "$RESULTS_IN_DRIVE/"
!rsync -a "$DATA_DIR/intermediate" "$RESULTS_IN_DRIVE/"
print("结果已回传：", RESULTS_IN_DRIVE)